# Fine-tune GLM-OCR for Vietnamese Diacritical Marks (v2)

Follows the official guide: [examples/finetune/README.md](https://github.com/zai-org/GLM-OCR/blob/main/examples/finetune/README.md)

**Dataset format:** ShareGPT with `messages`/`role`/`content` + `images` fields.

**Workflow:**
1. Chạy các bước 1→6 (setup, chỉ 1 lần)
2. Bước 7a: Train epoch đầu tiên
3. Bước 8: Merge & test kết quả
4. Nếu chưa ổn → chạy bước 7b (thêm 1 epoch) → quay lại bước 8 test
5. Lặp cho đến khi hài lòng → bước 10 save

**Requirements:**
- GPU: T4 (16GB) or better
- Upload `vietnamese_ocr.zip` to Google Drive `My Drive` root
  - Structure: `vietnamese_ocr/vietnamese_ocr.json` + `vietnamese_ocr/images/txt_*.png`

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Mount Drive & Extract Dataset

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Restore LoRA checkpoint from Drive (neu bi disconnect)
import os
drive_ckpt = "/content/drive/My Drive/glm-ocr-vn-checkpoints"
local_ckpt = "/content/glm-ocr-lora-sft"

if os.path.exists(drive_ckpt):
    os.makedirs(local_ckpt, exist_ok=True)
    !cp -r "{drive_ckpt}"/* "{local_ckpt}"/
    ckpts = [d for d in os.listdir(local_ckpt) if d.startswith("checkpoint-")]
    if ckpts:
        print(f"Restored {len(ckpts)} checkpoint(s):")
        for c in sorted(ckpts):
            print(f"  {c}")
    else:
        print("No checkpoints found in Drive. Will train from scratch.")
else:
    print("No saved checkpoints on Drive. Will train from scratch.")

In [ ]:
# Extract dataset
!cp "/content/drive/My Drive/vietnamese_ocr.zip" /content/
!cd /content && unzip -q -o vietnamese_ocr.zip
!echo "=== Structure ==="
!ls /content/vietnamese_ocr/
!echo "Images:" 0
!echo "JSON:" 

In [ ]:
# Verify dataset format: must have messages/role/content
import json, os

with open("/content/vietnamese_ocr/vietnamese_ocr.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Total samples: {len(data)}")
print(f"\nSample 0:")
print(json.dumps(data[0], ensure_ascii=False, indent=2))

# Verify format
sample = data[0]
assert "messages" in sample, "ERROR: missing 'messages' key"
assert "images" in sample, "ERROR: missing 'images' key"
assert sample["messages"][0]["role"] == "user", "ERROR: first message must be user"
assert sample["messages"][1]["role"] == "assistant", "ERROR: second message must be assistant"
assert "<image>" in sample["messages"][0]["content"], "ERROR: user message must contain <image>"
print("\n✓ Dataset format is correct!")

# Check image exists
img_path = os.path.join("/content/vietnamese_ocr", sample["images"][0])
print(f"\nImage path: {img_path}")
print(f"Image exists: {os.path.exists(img_path)}")

## 3. Install LLaMA-Factory

In [ ]:
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
%cd /content/LLaMA-Factory
!pip install -e ".[torch,metrics]" 2>&1 | tail -5

In [ ]:
# Pin transformers to 5.6.0 — compatible with both LLaMA-Factory (>=4.55, <=5.6.0) and GLM-OCR (>=5.3.0)
!pip install transformers==5.6.0 2>&1 | tail -3

In [ ]:
!llamafactory-cli version

## 4. Download GLM-OCR Model

In [ ]:
from huggingface_hub import snapshot_download

model_dir = snapshot_download(
    "zai-org/GLM-OCR",
    local_dir="/content/GLM-OCR",
    local_dir_use_symlinks=False,
)
print(f"Model downloaded to: {model_dir}")

## 5. Prepare Dataset for LLaMA-Factory

Copy dataset into `LLaMA-Factory/data/` (image paths are relative to this directory).

In [ ]:
# Copy dataset into LLaMA-Factory/data/
!cp /content/vietnamese_ocr/vietnamese_ocr.json /content/LLaMA-Factory/data/
!cp -r /content/vietnamese_ocr/images /content/LLaMA-Factory/data/images
!echo "Images:" 0
!echo "JSON:" 

In [ ]:
# Verify image path resolution
import json, os

with open("/content/LLaMA-Factory/data/vietnamese_ocr.json", "r") as f:
    data = json.load(f)

sample = data[0]
img_rel = sample["images"][0]
img_abs = os.path.join("/content/LLaMA-Factory/data", img_rel)
print(f"Image relative path: {img_rel}")
print(f"Image absolute path: {img_abs}")
print(f"Exists: {os.path.exists(img_abs)}")

# Check all images exist
missing = 0
for item in data:
    for img in item["images"]:
        if not os.path.exists(os.path.join("/content/LLaMA-Factory/data", img)):
            missing += 1
print(f"\nMissing images: {missing}/{len(data)}")

In [ ]:
# Register dataset in dataset_info.json
import json

ds_info_path = "/content/LLaMA-Factory/data/dataset_info.json"
with open(ds_info_path, "r") as f:
    info = json.load(f)

info["vietnamese_ocr"] = {
    "file_name": "vietnamese_ocr.json",
    "formatting": "sharegpt",
    "columns": {
        "messages": "messages",
        "images": "images"
    },
    "tags": {
        "role_tag": "role",
        "content_tag": "content",
        "user_tag": "user",
        "assistant_tag": "assistant"
    }
}

with open(ds_info_path, "w") as f:
    json.dump(info, f, indent=2, ensure_ascii=False)

print("✓ Dataset registered in dataset_info.json")
print(json.dumps(info["vietnamese_ocr"], indent=2))

## 6. Write Training Config

In [ ]:
yaml_content = """
### model
model_name_or_path: /content/GLM-OCR
trust_remote_code: true

### method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 32
lora_target: all

### dataset
dataset: vietnamese_ocr
template: glm_ocr
cutoff_len: 2048
preprocessing_num_workers: 8
dataloader_num_workers: 2
val_size: 0.1
per_device_eval_batch_size: 1
eval_strategy: steps
eval_steps: 100

### output
output_dir: /content/glm-ocr-lora-sft
logging_steps: 10
save_steps: 500
plot_loss: true
overwrite_output_dir: true
save_only_model: false
report_to: none

### train
per_device_train_batch_size: 4
gradient_accumulation_steps: 4
learning_rate: 1.0e-4
num_train_epochs: 1
lr_scheduler_type: cosine
warmup_ratio: 0.1
fp16: true
"""

with open("/content/glm_ocr_vn_lora_sft.yaml", "w") as f:
    f.write(yaml_content)

print("Config written to /content/glm_ocr_vn_lora_sft.yaml")
print(yaml_content)

---

## 7. Train (nhan lai de train them epoch)

Moi lan chay = them 1 epoch. Tu detect checkpoint cu de resume.
- **Lan dau:** train epoch 1 tu dau
- **Lan sau:** resume tu checkpoint cu, train them 1 epoch

Mat khoang 2 phut/epoch tren T4. Checkpoint duoc tu luu len Drive sau khi train xong.

In [ ]:
# Reset: xoa toan bo checkpoint de train tu dau
!rm -rf /content/glm-ocr-lora-sft/checkpoint-*
!rm -rf "/content/drive/My Drive/glm-ocr-vn-checkpoints"
print("Deleted all checkpoints. Next train will start from epoch 1.")

In [ ]:
import os, json

# Detect checkpoint
ckpt_dir = "/content/glm-ocr-lora-sft"
checkpoints = sorted([d for d in os.listdir(ckpt_dir) if d.startswith("checkpoint-")]) if os.path.exists(ckpt_dir) else []

if checkpoints:
    last_ckpt = os.path.join(ckpt_dir, checkpoints[-1])
    step = int(checkpoints[-1].split("-")[1])
    with open("/content/LLaMA-Factory/data/vietnamese_ocr.json", "r") as f:
        _ds = json.load(f)
    steps_per_epoch = int(len(_ds) * 0.9) // (4 * 4)
    next_epoch = step // steps_per_epoch + 1
    total_epochs = next_epoch + 1
    print(f"Resuming from {checkpoints[-1]} (epoch {next_epoch}) -> training to epoch {total_epochs}")
else:
    last_ckpt = None
    total_epochs = 1
    print("No checkpoint found -> training epoch 1")

# Write YAML
yaml_content = f"""
### model
model_name_or_path: /content/GLM-OCR
trust_remote_code: true

### method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 32
lora_target: all

### dataset
dataset: vietnamese_ocr
template: glm_ocr
cutoff_len: 2048
preprocessing_num_workers: 8
dataloader_num_workers: 2
val_size: 0.1
per_device_eval_batch_size: 1
eval_strategy: steps
eval_steps: 100

### output
output_dir: /content/glm-ocr-lora-sft
logging_steps: 10
save_steps: 500
plot_loss: true
overwrite_output_dir: false
save_only_model: false
report_to: none

### train
per_device_train_batch_size: 4
gradient_accumulation_steps: 4
learning_rate: 1.0e-4
num_train_epochs: {total_epochs}
lr_scheduler_type: cosine
warmup_ratio: 0.1
fp16: true
"""
if last_ckpt:
    yaml_content += f"resume_from_checkpoint: {last_ckpt}\n"

with open("/content/glm_ocr_vn_lora_sft.yaml", "w") as f:
    f.write(yaml_content)

os.environ["DISABLE_VERSION_CHECK"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
!DISABLE_VERSION_CHECK=1 CUDA_VISIBLE_DEVICES=0
!llamafactory-cli train /content/glm_ocr_vn_lora_sft.yaml

In [ ]:
# Auto-save checkpoint to Drive
!mkdir -p "/content/drive/My Drive/glm-ocr-vn-checkpoints"
!cp -r /content/glm-ocr-lora-sft/checkpoint-* "/content/drive/My Drive/glm-ocr-vn-checkpoints/"
print("Saved checkpoints to Drive")
!ls "/content/drive/My Drive/glm-ocr-vn-checkpoints/"

## 8. Merge & Quick Test

Sau mỗi epoch, merge LoRA weights và test nhanh trên 1 ảnh.

In [ ]:
# Merge LoRA weights
!llamafactory-cli export \
  --model_name_or_path /content/GLM-OCR \
  --adapter_name_or_path /content/glm-ocr-lora-sft \
  --template glm_ocr \
  --export_dir /content/glm-ocr-vn-merged \
  --trust_remote_code true

In [ ]:
# Quick test — OCR 1 ảnh từ dataset (theo cách dùng chính thức của GLM-OCR)
from PIL import Image
import json, os, glob

# Lấy 1 ảnh test
test_images = sorted(glob.glob("/content/vietnamese_ocr/images/txt_*.png"))
test_img = test_images[0]

# Hiển thị ảnh
img = Image.open(test_img)
display(img)

# Load merged model — dùng AutoModelForImageTextToText (không phải AutoModelForCausalLM)
from transformers import AutoProcessor, AutoModelForImageTextToText

model_path = "/content/glm-ocr-vn-merged"
processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(model_path, trust_remote_code=True, torch_dtype="auto", device_map="auto")

# Build input — dùng structured content format (không dùng <image> string)
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": test_img},
            {"type": "text", "text": "Text Recognition:"},
        ],
    }
]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

# Remove token_type_ids nếu có (GLM-OCR không dùng)
inputs.pop("token_type_ids", None)

# Generate
generated_ids = model.generate(**inputs, max_new_tokens=512, do_sample=False)

# Decode
result = processor.decode(generated_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(f"OCR Result:{result}")

# Ground truth
with open("/content/vietnamese_ocr/vietnamese_ocr.json", "r") as f:
    dataset = json.load(f)
fname = os.path.basename(test_img)
for item in dataset:
    if fname in item["images"][0]:
        print(f"Ground truth:{item['messages'][1]['content']}")
        break

**Danh gia ket qua:**
- Neu OCR dung dau tieng Viet -> chuyen xuong **buoc 10** de save
- Neu chua on -> quay lai **buoc 7** train them epoch, roi quay lai **buoc 8** test lai

---

## 10. Save to Google Drive

In [ ]:
!mkdir -p "/content/drive/My Drive/glm-ocr-vn"
!cp -r /content/glm-ocr-vn-merged/* "/content/drive/My Drive/glm-ocr-vn/"
print("✓ Model saved to Drive: My Drive/glm-ocr-vn/")